# RAG Pipeline: Small Language Model Benchmark for Urdu Tourism QA

This notebook reproduces the retrieval-augmented generation (RAG) evaluation pipeline used in
*"Bridging Resource Constraints in Low-Resource Language Question Answering: A Retrieval-Augmented
Small Language Models Benchmark"*.

It evaluates five small language models (Qwen2.5-1.5B, Qwen3-1.7B, Gemma-2-2B, Gemma-4-E2B,
TinyAya-Fire) on a 150-question locked Urdu tourism QA evaluation set, under both baseline
(no retrieval) and RAG (retrieval-augmented) conditions.

**Before running:**
- This notebook was built for Kaggle. It expects a Kaggle GPU environment and a Hugging Face
  token stored as a Kaggle Secret named `HF_TOKEN` (required for gated models: Gemma, Aya).
- Update `CONTEXT_CSV_PATH` and `LOCKED_PATH` (below) to point to your own copy of the corpus
  and locked evaluation set if you're not using the original Kaggle dataset paths.
- To reproduce a specific model's results, set `ACTIVE_MODEL` in the model-loading cell and run
  all cells top to bottom. Repeat once per model to reproduce all five.
- Package versions are pinned for reproducibility (see the first code cell). Do not remove the
  version pins — `transformers` in particular releases new versions roughly every 1–2 weeks,
  and unpinned installs will not reproduce the reported results reliably.

In [ ]:
!pip install -q "transformers==5.15.0"
!pip install -q "sentence-transformers==5.4.0"
!pip install -q "bert-score==0.3.12"
!pip install -q "faiss-gpu==1.15.0"
!pip install -q "sympy>=1.13" --upgrade

In [ ]:
import warnings, logging, os
warnings.filterwarnings("ignore")
logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)
logging.getLogger("bert_score").setLevel(logging.ERROR)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

## Authentication

Gemma and TinyAya-Fire are gated models on Hugging Face and require an accepted license
agreement plus an access token. Before running this cell:

1. Accept the license terms on the Hugging Face model pages for
   [`google/gemma-2-2b-it`](https://huggingface.co/google/gemma-2-2b-it),
   [`google/gemma-4-E2B-it`](https://huggingface.co/google/gemma-4-E2B-it), and
   [`CohereLabs/tiny-aya-fire`](https://huggingface.co/CohereLabs/tiny-aya-fire).
2. Generate a Hugging Face access token at
   [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).
3. In Kaggle, go to **Add-ons → Secrets** and add it as a secret named `HF_TOKEN`.

If you're running this outside Kaggle, replace the cell below with your own token-loading method
(e.g. an environment variable or `huggingface_hub.login()` with the token passed directly).

In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

hf_token = UserSecretsClient().get_secret("HF_TOKEN")
login(token=hf_token)

## Model Loading

Set `ACTIVE_MODEL` below to choose which of the five evaluated models to load. This notebook
evaluates one model per run — to reproduce results for all five, change `ACTIVE_MODEL` and
re-run the notebook from this cell onward, once per model.

| `ACTIVE_MODEL` value | Model |
|---|---|
| `"qwen"` | Qwen2.5-1.5B-Instruct |
| `"qwen3"` | Qwen3-1.7B |
| `"gemma"` | Gemma-2-2B-it |
| `"gemma_e2b"` | Gemma-4-E2B-it |
| `"aya"` | TinyAya-Fire |

Gemma-4-E2B uses a distinct model class (`Gemma4ForConditionalGeneration`) and loads via
`AutoProcessor` rather than `AutoTokenizer`, since it is a multimodal-capable model accessed
here in text-only mode.

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_CONFIGS = {
    "qwen":      "Qwen/Qwen2.5-1.5B-Instruct",
    "qwen3":     "Qwen/Qwen3-1.7B",
    "gemma":     "google/gemma-2-2b-it",
    "gemma_e2b": "google/gemma-4-E2B-it",
    "aya":       "CohereLabs/tiny-aya-fire",
}
ACTIVE_MODEL = "qwen"  # "qwen" | "qwen3" | "gemma" | "gemma_e2b" | "aya" choose as per your need and want
model_name = MODEL_CONFIGS[ACTIVE_MODEL]
print(f"Loading: {model_name}")

if ACTIVE_MODEL == "gemma_e2b":
    from transformers import AutoProcessor, Gemma4ForConditionalGeneration
    tokenizer = AutoProcessor.from_pretrained(model_name)  # processor stands in for tokenizer below
    model = Gemma4ForConditionalGeneration.from_pretrained(
        model_name, dtype=torch.bfloat16, device_map="auto",
    )
else:
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(
        model_name, dtype=torch.bfloat16, device_map="auto",
    )
print(f"Model loaded: {model.config._name_or_path}")

## Prompt Rules and Generation Configuration

Both baseline and RAG conditions use the same rule set and decoding configuration, differing
only in whether retrieved context is supplied — the sole reason for the manuscript's controlled
baseline-vs-RAG comparison to be valid (Section 4.3).

All five models use identical decoding settings: greedy decoding (`do_sample=False`),
`max_new_tokens=100`, `repetition_penalty=1.1`, `no_repeat_ngram_size=0` (Section 5.1).

Prompts instruct the model to answer exclusively in Urdu script, avoiding Devanagari, Latin
transliteration, or code-switching, and to withhold speculation when the RAG context does not
contain the answer.

In [ ]:
import contextlib, io, re

# gemma_e2b's "tokenizer" above is really an AutoProcessor; the real tokenizer lives at .tokenizer
tok = tokenizer.tokenizer if ACTIVE_MODEL == "gemma_e2b" else tokenizer

if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
    tok.pad_token_id = tok.eos_token_id  # Ensures absolute compatibility with fast tokenizers

model.generation_config.max_length = None
model.generation_config.max_new_tokens = None

eos_ids = [tok.eos_token_id]
if ACTIVE_MODEL in ("qwen", "qwen3"):
    im_end_id = tok.convert_tokens_to_ids("<|im_end|>")
    if im_end_id not in [None, tok.unk_token_id]:
        eos_ids.append(im_end_id)
elif ACTIVE_MODEL in ("gemma", "gemma_e2b"):
    end_of_turn_id = tok.convert_tokens_to_ids("<end_of_turn>")
    if end_of_turn_id not in [None, tok.unk_token_id]:
        eos_ids.append(end_of_turn_id)

if ACTIVE_MODEL == "aya":
    from transformers import pipeline as hf_pipeline
    aya_pipe = hf_pipeline("text-generation", model=model, tokenizer=tokenizer, return_full_text=False)

RULES_TEMPLATE = (
    "Rules:\n"
    "1. CRITICAL: The ENTIRE answer must be in Urdu script (Arabic/Naskh) only. "
    "Any Hindi/Devanagari script (e.g. गलगत is WRONG, گلگت is correct), English words, "
    "or code-switching anywhere in the answer makes the response invalid.\n"
    "2. Answer fully and completely — include all information needed to correctly "
    "answer the question, especially for questions requiring multiple facts or steps. "
    "Do not pad the answer with unnecessary or repeated information.\n"
    "3. Do not repeat the question. Do not explain your reasoning.\n"
    "4. Do not hallucinate or add information beyond what is given to you.\n"
    "5. Do not use markdown formatting (no **, no bullet points, no headers).\n"
    "6. Output nothing except the final Urdu answer. No commentary, no translation, no alternate scripts.\n"
)

RULES = {
    "qwen": RULES_TEMPLATE,
    "qwen3": RULES_TEMPLATE,
    "gemma": RULES_TEMPLATE,
    "gemma_e2b": RULES_TEMPLATE,
    "aya": RULES_TEMPLATE,
}

SOURCE_LINE = {
    "baseline": {
        m: "You are a question-answering assistant about Pakistani tourism. Answer the user's question directly using your own knowledge.\n\n"
        for m in RULES
    },
    "rag": {
        m: "You are a question-answering assistant about Pakistani tourism. Answer the question directly using ONLY the provided context below. If the answer is not in the context, respond only: معلومات دستیاب نہیں۔\n\n"
        for m in RULES
    },
}

PROMPTS = {
    cond: {m: SOURCE_LINE[cond][m] + RULES[m] for m in RULES}
    for cond in ["baseline", "rag"]
}

_GEN_PARAMS = {
    "qwen":      {"repetition_penalty": 1.1, "no_repeat_ngram_size": 0},
    "qwen3":     {"repetition_penalty": 1.1, "no_repeat_ngram_size": 0},
    "gemma":     {"repetition_penalty": 1.1, "no_repeat_ngram_size": 0},
    "gemma_e2b": {"repetition_penalty": 1.1, "no_repeat_ngram_size": 0},
    "aya":       {"repetition_penalty": 1.1, "no_repeat_ngram_size": 0},
}

## Baseline Generation

Generates an answer using only the model's parametric knowledge (no retrieved context).
TinyAya-Fire uses a `pipeline`-based call path; the other four models use `model.generate()`
directly. Both paths use identical decoding settings — `do_sample=False`, `repetition_penalty=1.1`,
`no_repeat_ngram_size=0`, `max_new_tokens=100` — sampling-only parameters (`temperature`, `top_p`,
`top_k`) are omitted throughout, since they have no effect under greedy decoding.

In [ ]:
import contextlib
import io
import re
import torch

@torch.inference_mode()
def ask_baseline(question):
    system = PROMPTS["baseline"][ACTIVE_MODEL]
    params = _GEN_PARAMS[ACTIVE_MODEL]

    # AYA (uses pipeline & returns early)
    if ACTIVE_MODEL == "aya":
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ]

        prompt = tok.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        out = aya_pipe(
            prompt,
            max_new_tokens=100,
            max_length=None,
            do_sample=False,
            repetition_penalty=params["repetition_penalty"],
            no_repeat_ngram_size=params["no_repeat_ngram_size"],
            return_full_text=False
        )

        return out[0]["generated_text"].strip()

    # GEMMA 4 E2B
    elif ACTIVE_MODEL == "gemma_e2b":
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": question}
        ]

    # GEMMA 2 (standard Gemma requires no system role in template)
    elif ACTIVE_MODEL == "gemma":
        messages = [{"role": "user", "content": f"{system}\n\n{question}"}]

    # QWEN3
    elif ACTIVE_MODEL == "qwen3":
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ]

    # QWEN2.5
    else:
        messages = [
            {"role": "system", "content": system},
            {"role": "user", "content": question},
        ]

    # Generate chat template text
    chat_kwargs = {"enable_thinking": False} if ACTIVE_MODEL == "qwen3" else {}
    text_input = tok.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True, **chat_kwargs
    )

    # Tokenize input using the resolved tokenizer 'tok'
    model_inputs = tok([text_input], return_tensors="pt").to(model.device)

    # Generate response
    with contextlib.redirect_stderr(io.StringIO()):
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=eos_ids,
            repetition_penalty=params["repetition_penalty"],
            no_repeat_ngram_size=params["no_repeat_ngram_size"],
        )

    # Strip the prompt from the generated tokens
    generated_ids = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    text = tok.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Clean up reasoning/thinking blocks if Qwen3 still outputs them
    if ACTIVE_MODEL == "qwen3":
        text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL).strip()

    return text


print(f"Active model: {ACTIVE_MODEL}")
print(ask_baseline("وادی ہنزہ کہاں واقع ہے؟"))

## Corpus Loading

Loads the manually curated Urdu tourism corpus (231 semantically chunked passages, Section 3.2)
and writes it to a clean JSON file for downstream embedding and retrieval.

**Path note:** `CONTEXT_CSV_PATH` below points to a private Kaggle dataset. If you're running
this notebook outside the original Kaggle environment, update this path to point to your own
copy of the corpus CSV (`id`, `text` columns required). The corpus itself is not publicly
released due to unresolved copyright status on portions of the underlying travel-blog source
material (Section 3.2); the 150-item locked evaluation set is released separately.

In [ ]:
import pandas as pd, json

# Update this path to point to your local copy of the corpus CSV (columns: id, text)
CONTEXT_CSV_PATH = "/kaggle/input/datasets/kiranf31/ragpipeline/context_urdu(Final)(Sheet1) (2).csv"

df_ctx = pd.read_csv(CONTEXT_CSV_PATH)
df_ctx.columns = df_ctx.columns.str.strip()
if "content" in df_ctx.columns and "text" not in df_ctx.columns:
    df_ctx = df_ctx.rename(columns={"content": "text"})
assert "id" in df_ctx.columns and "text" in df_ctx.columns
df_ctx = df_ctx[["id", "text"]].dropna(subset=["text"]).reset_index(drop=True)
df_ctx["id"] = df_ctx["id"].astype(int)
print(f"Clean context entries: {len(df_ctx)}")
data = df_ctx.to_dict(orient="records")
with open("rag_dataset.json", "w", encoding="utf-8") as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

## Embedding & Retrieval Index

Builds a dense vector index over the corpus using `intfloat/multilingual-e5-base` (Section 4.2).
Following the model's recommended encoding convention, passages are prefixed with `"passage: "`.
All embeddings are L2-normalized, so inner-product search in a FAISS `IndexFlatIP` index is
equivalent to cosine similarity — this allows exact nearest-neighbor retrieval without an
external vector database or network connectivity.

In [ ]:
import json, numpy as np, faiss
from sentence_transformers import SentenceTransformer

with open("rag_dataset.json", "r", encoding="utf-8") as f:
    data = json.load(f)

texts = ["passage: " + entry["text"] for entry in data]
embed_model = SentenceTransformer("intfloat/multilingual-e5-base")
embeddings = embed_model.encode(texts, convert_to_numpy=True, normalize_embeddings=True)

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)
index.add(embeddings)
print(f"FAISS index ready: {index.ntotal} vectors, dim={dim}")

## RAG Generation

Generates an answer using retrieved context from the corpus. The retriever returns the top-5
candidate chunks, discards any invalid FAISS indices (`-1`, which can occur if fewer than the
requested number of neighbors are available), and keeps the top-3 valid chunks as context
(Section 4.2). Decoding configuration is identical to the baseline condition — the only
difference between conditions is the presence of retrieved context in the prompt.

In [ ]:
@torch.inference_mode()
def rag_query(query, top_k=5, final_k=3):
    # Ensure we only retrieve up to what we actually need from FAISS
    q_emb = embed_model.encode(["query: " + query], convert_to_numpy=True, normalize_embeddings=True)
    D, I = index.search(q_emb, top_k)

    # Bugfix: Filter out invalid indices (-1) before lookup to prevent crash/corrupted context
    valid_indices = [idx for idx in I[0] if idx >= 0][:final_k]
    retrieved_texts = [data[i]["text"] for i in valid_indices]
    context = "\n\n".join(retrieved_texts)
    system = PROMPTS["rag"][ACTIVE_MODEL]
    params = _GEN_PARAMS[ACTIVE_MODEL]
    user_content = f"Context:\n{context}\n\nQuestion: {query}\n\nAnswer in Urdu script only, no other language or script:"

    # AYA (uses pipeline)
    if ACTIVE_MODEL == "aya":
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user_content}]
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        out = aya_pipe(
            prompt,
            max_new_tokens=100,
            max_length=None,
            do_sample=False,
            repetition_penalty=params["repetition_penalty"],
            no_repeat_ngram_size=params["no_repeat_ngram_size"],
            return_full_text=False
        )
        return out[0]["generated_text"].strip(), context

    # GEMMA 2
    if ACTIVE_MODEL == "gemma":
        messages = [{"role": "user", "content": f"{system}\n\n{user_content}"}]
    # GEMMA 4 E2B — keep the system instruction in its proper system boundary
    elif ACTIVE_MODEL == "gemma_e2b":
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system}]},
            {"role": "user", "content": [{"type": "text", "text": user_content}]}
        ]
    # QWEN & QWEN3
    else:
        messages = [{"role": "system", "content": system}, {"role": "user", "content": user_content}]

    chat_kwargs = {"enable_thinking": False} if ACTIVE_MODEL == "qwen3" else {}
    text_input = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, **chat_kwargs)
    model_inputs = tok([text_input], return_tensors="pt").to(model.device)

    with contextlib.redirect_stderr(io.StringIO()):
        generated_ids = model.generate(
            **model_inputs,
            max_new_tokens=100,
            do_sample=False,
            pad_token_id=tok.pad_token_id,
            eos_token_id=eos_ids,
            repetition_penalty=params["repetition_penalty"],
            no_repeat_ngram_size=params["no_repeat_ngram_size"],
        )

    generated_ids = [out[len(inp):] for inp, out in zip(model_inputs.input_ids, generated_ids)]
    answer = tok.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()
    if ACTIVE_MODEL == "qwen3":
        answer = re.sub(r"<think>.*?</think>", "", answer, flags=re.DOTALL).strip()
    return answer, context


# quick sanity check before i proceed
ans, ctx = rag_query("وادی ہنزہ کہاں واقع ہے؟")
print("Answer:", ans)

## Locked Evaluation Set

Loads the 150-question stratified evaluation set (50 easy / 50 medium / 50 hard, Section 3.3.2)
used for all baseline-vs-RAG comparisons in the paper. This file is released publicly alongside
this notebook — see the repository README for the download link.

**Path note:** update `LOCKED_PATH` below to point to your local copy of the eval set CSV
(columns: `id`, `question`, `true_answer`, `difficulty`).

In [ ]:
import pandas as pd

# Update this path to point to your local copy of the locked evaluation set
LOCKED_PATH = "/kaggle/input/datasets/kiranf31/locked-evaluation-set/eval_150_general_locked.csv"
df_locked = pd.read_csv(LOCKED_PATH, encoding="utf-8")

eval_data = [
    {
        "id": row["id"],
        "question": row["question"],
        "true_answer_full": row["true_answer"],
        "difficulty": row["difficulty"].strip().lower(),
    }
    for _, row in df_locked.iterrows()
]

print(f"Loaded locked eval set: {len(eval_data)} questions")
print("Difficulty counts:", {d: sum(1 for e in eval_data if e["difficulty"] == d) for d in ["easy", "medium", "hard"]})

## Generation

Runs baseline and RAG generation for every question in the locked evaluation set, using the
model selected via `ACTIVE_MODEL` above.

Progress is checkpointed periodically to `results_{ACTIVE_MODEL}_150.json`. The checkpoint
stores a hash of the evaluation set alongside the results — if you edit `eval_data` (e.g. to
correct a question) and re-run this cell, the mismatched hash automatically invalidates the old
checkpoint and regenerates from scratch, rather than silently resuming with stale answers for
changed questions.

In [ ]:
import json, hashlib, os

# Hash the current eval set so we can detect if it changed since the last checkpoint
eval_data_str = json.dumps(eval_data, sort_keys=True, ensure_ascii=False)
eval_hash = hashlib.md5(eval_data_str.encode("utf-8")).hexdigest()

CKPT_PATH = f"results_{ACTIVE_MODEL}_150.json"
results = []
done_ids = set()

if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH, "r", encoding="utf-8") as f:
        ckpt = json.load(f)
    if ckpt.get("eval_hash") == eval_hash:
        results = ckpt["results"]
        done_ids = {r["id"] for r in results}
        print(f"Resuming: {len(results)} already done (eval set unchanged).")
    else:
        print("Eval set has changed since last checkpoint — starting fresh.")

for i, e in enumerate(eval_data):
    if e["id"] in done_ids:
        continue
    base_ans = ask_baseline(e["question"])
    rag_ans, context = rag_query(e["question"])
    results.append({
        "id": e["id"], "model": ACTIVE_MODEL, "question": e["question"],
        "true_answer": e["true_answer_full"],
        "baseline_answer": base_ans, "rag_answer": rag_ans,
        "difficulty": e["difficulty"],
        "context": context,
    })
    print(f"[{i+1}/{len(eval_data)}] [{e['difficulty'].upper()}] ID: {e['id']}")
    print(f"Q:            {e['question']}")
    print(f"Ground Truth: {e['true_answer_full']}")
    print(f"Baseline:     {base_ans}")
    print(f"RAG:          {rag_ans}")
    print("-" * 60)
    if (i + 1) % 25 == 0:
        with open(CKPT_PATH, "w", encoding="utf-8") as f:
            json.dump({"eval_hash": eval_hash, "results": results}, f, ensure_ascii=False, indent=2)
        print(f"Checkpoint saved at {i+1}/{len(eval_data)}")

with open(CKPT_PATH, "w", encoding="utf-8") as f:
    json.dump({"eval_hash": eval_hash, "results": results}, f, ensure_ascii=False, indent=2)
print(f"Done. {len(results)} results saved to {CKPT_PATH}")

## Automatic Evaluation: BERTScore F1

Computes BERTScore F1 between generated answers and reference answers using two multilingual
encoders — mBERT (`bert-base-multilingual-cased`, via `lang="ur"`) and XLM-RoBERTa-large
(Section 5.2.1). Both use raw (non-rescaled) BERTScore (`rescale_with_baseline=False`); see the
paper's evaluation-metrics discussion for why raw scores run high and compressed relative to an
intuitive 0–1 quality scale.

**Note on `script_mismatch`:** the column below is a coarse diagnostic (does the generated answer
contain any Urdu-script characters at all?) and is *not* the script-contamination methodology
used for the paper's reported results, which specifically targets Devanagari-range characters
via human annotation (Table 6). This column is exploratory only and unused in reported results.

In [ ]:
from bert_score import score
import pandas as pd, re

baseline_answers = [r["baseline_answer"] for r in results]
rag_answers = [r["rag_answer"] for r in results]
true_answers = [r["true_answer"] for r in results]

# mBERT (lang="ur" routes to bert-base-multilingual-cased)
_, _, F1_base_mbert = score(baseline_answers, true_answers, lang="ur", rescale_with_baseline=False)
_, _, F1_rag_mbert  = score(rag_answers,      true_answers, lang="ur", rescale_with_baseline=False)

# XLM-RoBERTa-large
_, _, F1_base_xlm = score(baseline_answers, true_answers, model_type="xlm-roberta-large", rescale_with_baseline=False)
_, _, F1_rag_xlm  = score(rag_answers,      true_answers, model_type="xlm-roberta-large", rescale_with_baseline=False)

df = pd.DataFrame(results)
df["difficulty"] = pd.Categorical(df["difficulty"], categories=["easy", "medium", "hard"], ordered=True)

df["f1_baseline_mbert"] = F1_base_mbert.numpy()
df["f1_rag_mbert"]      = F1_rag_mbert.numpy()
df["f1_delta_mbert"]    = df["f1_rag_mbert"] - df["f1_baseline_mbert"]

df["f1_baseline_xlm"] = F1_base_xlm.numpy()
df["f1_rag_xlm"]      = F1_rag_xlm.numpy()
df["f1_delta_xlm"]    = df["f1_rag_xlm"] - df["f1_baseline_xlm"]

# Exploratory diagnostic only — see markdown note above
def is_urdu_script(s):
    return bool(re.search(r'[\u0600-\u06FF]', str(s)))
df["script_mismatch"] = df["true_answer"].apply(is_urdu_script) != df["rag_answer"].apply(is_urdu_script)
if df["script_mismatch"].sum():
    print(f"NOTE (diagnostic only): {df['script_mismatch'].sum()}/{len(df)} rows show a script_mismatch flag")

print(" mBERT: ")
print(f"Baseline F1: {df['f1_baseline_mbert'].mean():.4f} | RAG F1: {df['f1_rag_mbert'].mean():.4f} | Delta: {df['f1_delta_mbert'].mean():+.4f}")
for grp, sub in df.groupby("difficulty", observed=True):
    print(f"  [{grp}] base={sub['f1_baseline_mbert'].mean():.4f}  rag={sub['f1_rag_mbert'].mean():.4f}  delta={sub['f1_delta_mbert'].mean():+.4f}")

print("\n XLM-RoBERTa-large:")
print(f"Baseline F1: {df['f1_baseline_xlm'].mean():.4f} | RAG F1: {df['f1_rag_xlm'].mean():.4f} | Delta: {df['f1_delta_xlm'].mean():+.4f}")
for grp, sub in df.groupby("difficulty", observed=True):
    print(f"  [{grp}] base={sub['f1_baseline_xlm'].mean():.4f}  rag={sub['f1_rag_xlm'].mean():.4f}  delta={sub['f1_delta_xlm'].mean():+.4f}")

OUT_PATH = f"rag_evaluation_results_{ACTIVE_MODEL}_150.csv"
df.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"Saved to {OUT_PATH}")

## Visualization

Plots per-difficulty baseline-vs-RAG F1 scores and the corresponding improvement (Δ F1) for
both encoders, for the currently active model.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

difficulties = ["easy", "medium", "hard"]
x = np.arange(len(difficulties))
w = 0.3

fig, axes = plt.subplots(2, 2, figsize=(13, 10))
fig.suptitle(f"RAG vs Baseline — {ACTIVE_MODEL} (n={len(df)})", fontsize=14, fontweight="bold")

for row, metric, label in [(0, "xlm", "XLM-RoBERTa-large"), (1, "mbert", "mBERT")]:
    base_scores = [df[df["difficulty"] == d][f"f1_baseline_{metric}"].mean() for d in difficulties]
    rag_scores  = [df[df["difficulty"] == d][f"f1_rag_{metric}"].mean()      for d in difficulties]
    delta_scores = [df[df["difficulty"] == d][f"f1_delta_{metric}"].mean()  for d in difficulties]

    ax0, ax1 = axes[row][0], axes[row][1]

    bars1 = ax0.bar(x - w/2, base_scores, width=w, color="#B4B2A9", label="Baseline")
    bars2 = ax0.bar(x + w/2, rag_scores,  width=w, color="#378ADD", label="RAG")

    ax0.set_title(f"{label} — F1 by difficulty")
    ax0.set_xticks(x); ax0.set_xticklabels(difficulties)
    ax0.set_ylabel("F1 Score")
    max_score = max(base_scores + rag_scores)
    min_score = min(base_scores + rag_scores)
    ax0.set_ylim(max(0, min_score - 0.05), min(1.0, max_score + 0.05))
    ax0.legend()
    for bar in list(bars1) + list(bars2):
        ax0.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                  f"{bar.get_height():.3f}", ha="center", va="bottom", fontsize=9)

    bars3 = ax1.bar(difficulties, delta_scores, color="#5DCAA5", width=0.5)
    ax1.set_title(f"{label} — RAG improvement (Δ F1)")
    ax1.set_ylabel("Δ F1")
    ax1.axhline(0, color="black", linewidth=0.8)
    for bar, val in zip(bars3, delta_scores):
        ax1.text(bar.get_x() + bar.get_width()/2, val + 0.001, f"{val:+.3f}",
                  ha="center", va="bottom", fontsize=10, fontweight="bold")

plt.tight_layout()
OUT_CHART = f"/kaggle/working/rag_chart_{ACTIVE_MODEL}_150.png"
plt.savefig(OUT_CHART, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved chart to {OUT_CHART}")

## GPU Cleanup

Frees GPU memory before switching to the next model. Required if you're evaluating multiple
models in a single Kaggle session — re-run from the model-loading cell after this, with
`ACTIVE_MODEL` changed.

In [ ]:
import torch, gc

if 'model' in locals(): del model
if 'tokenizer' in locals(): del tokenizer
gc.collect()
torch.cuda.empty_cache()
if 'aya_pipe' in locals(): del aya_pipe
print(f"GPU cleared after {ACTIVE_MODEL}.")